# 02 Data Cleaning

## Purpose
Clean data types, parse dates, check missingness, and create a long diagnosis event table from primary and secondary diagnosis fields.

## Imports

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"
MODELS_DIR = PROJECT_ROOT / "models"
for directory in [FIGURES_DIR, TABLES_DIR, MODELS_DIR, DATA_DIR / "processed"]:
    directory.mkdir(parents=True, exist_ok=True)

import pandas as pd

from src.data_cleaning import load_raw_data, clean_all, make_condition_events, save_processed

## Data loading

In [2]:
raw = load_raw_data(DATA_DIR)
data = clean_all(raw)

## Missing values and data types

In [3]:
missing_tables = []
for name, df in data.items():
    missing = df.isna().sum().reset_index()
    missing.columns = ["column", "missing_count"]
    missing["missing_pct"] = missing["missing_count"] / len(df)
    missing["table"] = name
    missing_tables.append(missing)

missing_summary = pd.concat(missing_tables, ignore_index=True)
missing_summary.sort_values("missing_pct", ascending=False).head(20)

,column,missing_count,missing_pct,table
65,days_to_readmission,9358,0.85065,outcomes
0,patient_id,0,0.00000,patients
42,reference_high,0,0.00000,lab_results
48,dose,0,0.00000,medications
47,medication,0,0.00000,medications
46,patient_id,0,0.00000,medications
45,delta_from_normal,0,0.00000,lab_results
44,is_abnormal,0,0.00000,lab_results
43,flag,0,0.00000,lab_results
41,reference_low,0,0.00000,lab_results


In [4]:
for name, df in data.items():
    print(f"\n{name.upper()}")
    print(df.dtypes)


PATIENTS
patient_id                     object
age                           float64
sex                            object
bmi                           float64
systolic_bp                   float64
diastolic_bp                  float64
heart_rate                      int64
temperature_f                 float64
smoking_status                 object
alcohol_use                    object
exercise_level                 object
insurance_type                 object
charlson_index                  int64
dx_hypertension                 int64
dx_type2_diabetes               int64
dx_hyperlipidemia               int64
dx_obesity                      int64
dx_coronary_artery_disease      int64
dx_heart_failure                int64
dx_atrial_fibrillation          int64
dx_chronic_kidney_disease       int64
dx_copd                         int64
dx_asthma                       int64
dx_depression                   int64
dx_anxiety                      int64
dx_hypothyroidism               int64
dx

## Main cleaning step: condition events

In [5]:
condition_events = make_condition_events(data["diagnoses"])
condition_events.head()

,patient_id,visit_date,visit_type,condition,icd10,provider_specialty,diagnosis_position
0,P0000001,2018-07-26,telehealth,hyperlipidemia,E78.5,internal_medicine,primary
1,P0000001,2018-07-26,telehealth,chronic_kidney_disease,N18.3,internal_medicine,secondary
2,P0000001,2020-04-09,outpatient,hyperlipidemia,E78.5,nephrology,primary
3,P0000001,2020-04-09,outpatient,chronic_kidney_disease,N18.3,nephrology,secondary
4,P0000001,2020-04-09,outpatient,osteoarthritis,M19.90,nephrology,secondary


In [6]:
condition_events_summary = condition_events.groupby("diagnosis_position").agg(
    events=("condition", "count"),
    unique_patients=("patient_id", "nunique"),
    unique_conditions=("condition", "nunique"),
).reset_index()
condition_events_summary

,diagnosis_position,events,unique_patients,unique_conditions
0,primary,274592,100000,15
1,secondary,223236,74266,15


## Key findings

- Date columns are converted to datetime so the project can separate baseline and future windows.
- Primary and secondary diagnoses are converted into one condition-event table, which is easier for multimorbidity counts and disease co-occurrence analysis.
- Cleaning is intentionally simple and transparent for reproducibility.

## Saved outputs

In [7]:
condition_path = save_processed(condition_events, "condition_events.csv")
missing_summary.to_csv(TABLES_DIR / "02_missing_summary.csv", index=False)
condition_events_summary.to_csv(TABLES_DIR / "02_condition_events_summary.csv", index=False)
print("Saved", condition_path)

Saved /Users/chinmaynaringrekar/PythonProject/Disease Progression and Multimorbidity Prediction/data/processed/condition_events.csv
